# Simple models for IMDb sentiment classification

This notebook compares two basic classifiers:

- **Linear classifier:** one affine layer that maps a Bag-of-Words representation to two class scores.
- **MLP:** a hidden layer with ReLU, followed by an output layer.

The first goal is to establish a simple baseline and practise the PyTorch training workflow. We will start with the linear classifier.

## 1. Imports and project paths

The path setup below searches the current directory and its parents for the project-level `src` folder. This lets the notebook be run from a `notebooks/` directory or the project root.

In [24]:
import sys
from functools import partial
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader, random_split

# Find the project root (the directory containing src/).
current_dir = Path.cwd()
project_root = next(
    (path for path in [current_dir, *current_dir.parents] if (path / "src").is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError("Could not find the project root containing the 'src' directory.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DATA_ROOT = project_root / "data" / "aclImdb"
if not DATA_ROOT.exists():
    raise FileNotFoundError(f"IMDb dataset directory not found: {DATA_ROOT}")

## 2. Load the IMDb dataset

The dataset returns raw review text and an integer label (`1` for positive, `0` for negative). We keep the official test set separate and split the official training set into training and validation subsets.

In [25]:
from src.data.dataset import IMDbDataset
from src.data.tokenizer import WhitespaceTokenizer
from src.data.vocabulary import Vocabulary
from src.models.linear import LinearModel, LinearWithReluModel

train_dataset = IMDbDataset(DATA_ROOT, train=True)
test_dataset = IMDbDataset(DATA_ROOT, train=False)

print(f"Training examples available: {len(train_dataset)}")
print(f"Test examples: {len(test_dataset)}")

# Reserve 10% of the original training data for validation.
train_size = int(0.9 * len(train_dataset))
val_size = len(train_dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=generator
)

print(f"Training subset: {len(train_subset)}")
print(f"Validation subset: {len(val_subset)}")

Training examples available: 25000
Test examples: 25000
Training subset: 22500
Validation subset: 2500


## 3. Build the vocabulary

The vocabulary is built **only from the training subset**. This avoids using validation or test text when deciding which tokens belong to the vocabulary.

The vocabulary maps each token to an integer ID. Those IDs are useful for lookup, but they are not meaningful numerical features by themselves. We will convert them into Bag-of-Words vectors next.

In [26]:
tokenizer = WhitespaceTokenizer()
vocab = Vocabulary(tokenizer, min_freq=2)
vocab.build([review for review, _ in train_subset])

print(f"Vocabulary size (including special tokens): {len(vocab)}")

Vocabulary size (including special tokens): 102801


## 4. Convert reviews into Bag-of-Words vectors

A Bag-of-Words (BoW) vector has one position for every vocabulary ID. Its value is the number of times that token occurs in the review.

For example, with a vocabulary ordered as `[great, movie, bad]`, the review `great movie great` becomes `[2, 1, 0]`.

This representation discards word order. That is a limitation, but it gives the linear classifier a meaningful fixed-size input. We use raw token counts for this first baseline.

The BoW collate function returns:

- `inputs`: a float tensor of shape `[batch_size, vocab_size]`;
- `labels`: a long tensor of shape `[batch_size]`.

Unlike the sequence-based `collate_fn`, this function does not pad or truncate reviews. A BoW vector always has the same size because its length is the vocabulary size.

In [27]:
from src.data.preprocessing import bow_collate_fn

bow_collate = partial(bow_collate_fn, vocab=vocab)

train_loader = DataLoader(
    train_subset,
    batch_size=32,
    shuffle=True,
    collate_fn=bow_collate
)

val_loader = DataLoader(
    val_subset,
    batch_size=32,
    shuffle=False,
    collate_fn=bow_collate
)

# Inspect one batch.
inputs, labels = next(iter(train_loader))
print("Input shape:", inputs.shape)       # [32, len(vocab)]
print("Label shape:", labels.shape)       # [32]
print("Input dtype:", inputs.dtype)       # torch.float32
print("Label dtype:", labels.dtype)       # torch.int64 / torch.long
print("Nonzero features in first review:", torch.count_nonzero(inputs[0]).item())

Input shape: torch.Size([32, 102801])
Label shape: torch.Size([32])
Input dtype: torch.float32
Label dtype: torch.int64
Nonzero features in first review: 141


## 5. Define the models

The linear classifier computes `y = Wx + b`. Its input dimension must therefore be the vocabulary size, and its output dimension is 2 (negative and positive class logits).

The second model is a small **multilayer perceptron (MLP)**: a hidden linear layer, ReLU activation, and a final linear layer. The final layer returns raw logits; do not add Softmax when training with `CrossEntropyLoss`.

In [28]:
linear_model = LinearModel(input_dim=len(vocab), output_dim=2)

mlp_model = LinearWithReluModel(
    input_dim=len(vocab),
    hidden_dim=128,
    output_dim=2
)

criterion = nn.CrossEntropyLoss()
optimizer_linear = torch.optim.Adam(linear_model.parameters(), lr=0.001)
optimizer_mlp = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

## 6. Train the linear classifier

Each training batch follows the standard PyTorch sequence:

1. Clear gradients from the previous batch.
2. Run the forward pass to obtain logits.
3. Calculate the loss.
4. Backpropagate the loss to calculate gradients.
5. Update the model parameters.

`model.train()` sets training mode. It is good practice even though this particular model has no Dropout or BatchNorm layers.

In the same loop we also evaluate the model on the validation set after each epoch, using `model.eval()`

In [29]:
num_epochs = 5
linear_train_losses = []
linear_val_losses = []

for epoch in range(num_epochs):
    # Training
    linear_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer_linear.zero_grad()

        outputs = linear_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_linear.step()

        train_loss += loss.item()

    # Validation
    linear_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = linear_model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total

    linear_train_losses.append(avg_train_loss)
    linear_val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/5 | Train Loss: 0.3882 | Val Loss: 0.3130 | Val Accuracy: 0.8924
Epoch 2/5 | Train Loss: 0.1862 | Val Loss: 0.2936 | Val Accuracy: 0.8960
Epoch 3/5 | Train Loss: 0.1195 | Val Loss: 0.2860 | Val Accuracy: 0.8924
Epoch 4/5 | Train Loss: 0.0839 | Val Loss: 0.2874 | Val Accuracy: 0.8964
Epoch 5/5 | Train Loss: 0.0609 | Val Loss: 0.2915 | Val Accuracy: 0.8940


## 7. MLP with one Hidden Layer
Here we are repeating the same loop using the MLP as our model.

**Limitations of this baseline:** BoW ignores word order and context. For example, it cannot reliably distinguish `the movie was good` from `the movie was not good` based on word order alone.

In [30]:
num_epochs = 5
mlp_train_losses = []
mlp_val_losses = []

for epoch in range(num_epochs):
    # Training
    mlp_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer_mlp.zero_grad()

        outputs = mlp_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_mlp.step()

        train_loss += loss.item()

    # Validation
    mlp_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = mlp_model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total

    mlp_train_losses.append(avg_train_loss)
    mlp_val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/5 | Train Loss: 0.3423 | Val Loss: 0.3111 | Val Accuracy: 0.8824
Epoch 2/5 | Train Loss: 0.0672 | Val Loss: 0.4074 | Val Accuracy: 0.8784
Epoch 3/5 | Train Loss: 0.0112 | Val Loss: 0.5968 | Val Accuracy: 0.8764
Epoch 4/5 | Train Loss: 0.0027 | Val Loss: 0.6233 | Val Accuracy: 0.8776
Epoch 5/5 | Train Loss: 0.0021 | Val Loss: 0.7242 | Val Accuracy: 0.8776


Analyzing training and validation losses we can see that the linear model generalizes fairly well for its scale, but the MLP is overfitting the training data since the validation loss keeps increasing with the epochs. This experiment used an MLP with a single hidden layer containing 128 neurons.
We will try two routes to prevent overfitting:
- using an hidden layer with 32 neurons
- using weight decay as regularization

In [32]:
smaller_mlp_model = LinearWithReluModel(
    input_dim=len(vocab),
    hidden_dim=32,
    output_dim=2
)

criterion = nn.CrossEntropyLoss()
optimizer_smaller_mlp = torch.optim.Adam(smaller_mlp_model.parameters(), lr=0.001)

In [33]:
num_epochs = 5
smaller_mlp_train_losses = []
smaller_mlp_val_losses = []

for epoch in range(num_epochs):
    # Training
    smaller_mlp_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer_smaller_mlp.zero_grad()

        outputs = smaller_mlp_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_smaller_mlp.step()

        train_loss += loss.item()

    # Validation
    smaller_mlp_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = smaller_mlp_model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total

    smaller_mlp_train_losses.append(avg_train_loss)
    smaller_mlp_val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/5 | Train Loss: 0.3483 | Val Loss: 0.3036 | Val Accuracy: 0.8880
Epoch 2/5 | Train Loss: 0.0914 | Val Loss: 0.3504 | Val Accuracy: 0.8880
Epoch 3/5 | Train Loss: 0.0252 | Val Loss: 0.4233 | Val Accuracy: 0.8880
Epoch 4/5 | Train Loss: 0.0087 | Val Loss: 0.5106 | Val Accuracy: 0.8764
Epoch 5/5 | Train Loss: 0.0046 | Val Loss: 0.5596 | Val Accuracy: 0.8812


As we can see it is still overfitting the data, let's see how it goes adding regularization.

In [36]:
wd_smaller_mlp_model = LinearWithReluModel(
    input_dim=len(vocab),
    hidden_dim=32,
    output_dim=2
)

criterion = nn.CrossEntropyLoss()

optimizer_wd_smaller_mlp = torch.optim.Adam(
    wd_smaller_mlp_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [37]:
num_epochs = 5
wd_smaller_mlp_train_losses = []
wd_smaller_mlp_val_losses = []

for epoch in range(num_epochs):
    # Training
    wd_smaller_mlp_model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer_wd_smaller_mlp.zero_grad()

        outputs = wd_smaller_mlp_model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_wd_smaller_mlp.step()

        train_loss += loss.item()

    # Validation
    wd_smaller_mlp_model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = wd_smaller_mlp_model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total

    wd_smaller_mlp_train_losses.append(avg_train_loss)
    wd_smaller_mlp_val_losses.append(avg_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/5 | Train Loss: 0.3502 | Val Loss: 0.3012 | Val Accuracy: 0.8868
Epoch 2/5 | Train Loss: 0.1101 | Val Loss: 0.3743 | Val Accuracy: 0.8760
Epoch 3/5 | Train Loss: 0.0496 | Val Loss: 0.4789 | Val Accuracy: 0.8740
Epoch 4/5 | Train Loss: 0.0593 | Val Loss: 0.6603 | Val Accuracy: 0.8524
Epoch 5/5 | Train Loss: 0.0677 | Val Loss: 0.5434 | Val Accuracy: 0.8704
